### TASK 1

### 1.2 Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10
})

RANDOM_STATE = 42
print('Libraries loaded successfully.')

Libraries loaded successfully.


### 1.3 Load Dataset

In [2]:
df = pd.read_excel('Online_Retail.xlsx', engine='openpyxl')
print(f'Raw dataset shape: {df.shape}')
print('Column names:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Online_Retail.xlsx'

### 1.4 Data Pre-processing

In [ ]:
# Step 1: Check missing values and drop missing CustomerID
print('Missing values before cleaning:')
print(df.isnull().sum())
df_clean = df.dropna(subset=['CustomerID']).copy()
print(f'\nRows after dropping missing CustomerID: {len(df_clean):,}')

In [ ]:
# Step 2: Remove cancellations (InvoiceNo starts with C)
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
print(f'Rows after removing cancellations: {len(df_clean):,}')

# Step 3: Remove non-positive Quantity and UnitPrice
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
print(f'Rows after removing invalid Quantity/UnitPrice: {len(df_clean):,}')

# Step 4: Parse InvoiceDate
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Step 5: Derive TotalPrice
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

# CustomerID as integer
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

print('\nCleaned dataset sample:')
df_clean.head(3)

In [ ]:
# Step 6: Build RFM table
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Snapshot date (reference for Recency): {snapshot_date.date()}')

rfm = df_clean.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

print(f'RFM table shape: {rfm.shape}')
print('\nRFM summary statistics:')
print(rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2))

In [ ]:
# Step 7: Winsorise outliers at 99th percentile
for col in ['Recency', 'Frequency', 'Monetary']:
    cap = rfm[col].quantile(0.99)
    rfm[col] = rfm[col].clip(upper=cap)
    print(f'{col} capped at 99th percentile: {cap:.2f}')

# Step 8: Standardise
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

print('\nStandardisation complete.')
print(f'Mean per feature (should be ~0): {rfm_scaled.mean(axis=0).round(4)}')
print(f'Std per feature  (should be ~1): {rfm_scaled.std(axis=0).round(4)}')